In [1]:
import json
from pathlib import Path

In [2]:
data = []

with open("mpchat_hybrid_sentiment_updated.json", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        data.append(json.loads(line))

print(f"Loaded {len(data)} samples")


Loaded 7007 samples


In [3]:
import pprint

pprint.pprint(data[1])


{'context': [{'debug': {'alice_sentiment': 0.743,
                        'neg_score': 0.0,
                        'pos_score': 1.0,
                        'similarity': 0.3486178517341614},
              'sentiment_towards_target': 'positive',
              'speaker_id': 'TransparentCircle',
              'turn_index': 1,
              'utterance': 'did you do that with nescafe?...'}],
 'dialog_id': '57fb8901-322c-4086-bed7-2fcb74922cd6',
 's_node': {'speaker_id': 'Soggypaper_',
            'turn_index': 2,
            'utterance': 'yes, i did hahaha'},
 'target_speaker': 'Soggypaper_'}


In [42]:
def build_interaction_text(sample):
    """
    Converts one dataset sample into a clean, persona-focused interaction text.
    """
    ctx = sample["context"][-1]  # recency already enforced upstream
    s = sample["s_node"]

    interaction = f"""
Other user ({ctx['speaker_id']}) said
(sentiment toward {sample['target_speaker']}: {ctx['sentiment_towards_target']}):
"{ctx['utterance']}"

{sample['target_speaker']} replied:
"{s['utterance']}"
""".strip()

    return interaction


In [5]:
print(build_interaction_text(data[0]))

Other user (stylofylo) said
(sentiment toward nickeymousee: positive):
"omg i work at the cafe where we sell ur pins! they're so cute"

nickeymousee replied:
"which cafe? i only have them at trueshot right now and in edmonton soon edit: saw a comment for hamilton haha, so cool"


In [43]:
EMPTY_MEMORY = '{"facts": "", "style": ""}'

def build_memory_prompt(speaker, previous_memory, interaction_text):
    return (
        f"Speaker: {speaker}\n\n"
        f"Previous memory: {previous_memory if previous_memory else EMPTY_MEMORY}\n\n"
        f"New interaction:\n"
        f"{interaction_text}\n\n"
        f"Updated memory (JSON):\n"
        f"{{"
    )


In [7]:
prompt = build_memory_prompt(
    speaker=data[0]["target_speaker"],
    previous_memory="",
    interaction_text=build_interaction_text(data[0])
)

print(prompt)



SYSTEM:
You are a function that outputs JSON only.

USER:
Maintain a long-term memory about the speaker "nickeymousee".

Current memory (JSON):
{{"facts": "", "style": ""}}

New interaction:
Other user (stylofylo) said
(sentiment toward nickeymousee: positive):
"omg i work at the cafe where we sell ur pins! they're so cute"

nickeymousee replied:
"which cafe? i only have them at trueshot right now and in edmonton soon edit: saw a comment for hamilton haha, so cool"

TASK:
Extract persona information from the new interaction and update the memory.
You MAY extract facts that are directly stated or clearly implied by the speaker's own reply.
Do NOT describe the update process itself.


STRICT OUTPUT REQUIREMENTS:
- Output ONLY valid JSON
- Output EXACTLY two keys: "facts" and "style"
- Facts explicitly mentioned by the speaker (e.g., selling items, locations, work) are NOT considered invented
- Do NOT include explanations
- Do NOT include examples
- Do NOT include markdown
- Do NOT inclu

In [8]:
def call_llm(prompt):
    """
    Temporary stub for LLM call.
    Replace this with a real LLM later.
    """
    # For now, just return a deterministic placeholder
    return "The speaker appears friendly and engages positively with others about their work."


In [9]:
from collections import defaultdict

def run_recursive_summarization(dataset):
    memories = defaultdict(str)   # memory per target_speaker
    augmented_dataset = []

    for idx, sample in enumerate(dataset):
        speaker = sample["target_speaker"]

        interaction_text = build_interaction_text(sample)

        prompt = build_memory_prompt(
            speaker=speaker,
            previous_memory=memories[speaker],
            interaction_text=interaction_text
        )

        updated_memory = call_llm(prompt)

        # Update memory store
        memories[speaker] = updated_memory

        # Attach memory to sample
        new_sample = sample.copy()
        new_sample["memory_summary"] = updated_memory

        augmented_dataset.append(new_sample)

    return augmented_dataset, memories


In [10]:
test_data = data[:5]

augmented_data, final_memories = run_recursive_summarization(test_data)

for d in augmented_data:
    print("=" * 70)
    print("Target speaker:", d["target_speaker"])
    print("Memory summary:", d["memory_summary"])


Target speaker: nickeymousee
Memory summary: The speaker appears friendly and engages positively with others about their work.
Target speaker: Soggypaper_
Memory summary: The speaker appears friendly and engages positively with others about their work.
Target speaker: Soggypaper_
Memory summary: The speaker appears friendly and engages positively with others about their work.
Target speaker: Soggypaper_
Memory summary: The speaker appears friendly and engages positively with others about their work.
Target speaker: BeastaBubbles
Memory summary: The speaker appears friendly and engages positively with others about their work.


# Memory Length & Schema Enforcement

In [11]:
def clamp_text(text, max_sentences=3):
    """
    Ensures memory remains short and summary-like.
    """
    if not text:
        return ""

    sentences = [s.strip() for s in text.split(".") if s.strip()]
    return ". ".join(sentences[:max_sentences])

In [12]:
def stabilize_memory(prev, new):
    """
    Avoid unnecessary memory churn.
    """
    if not new:
        return prev
    if new.strip() == prev.strip():
        return prev
    return new

In [13]:
from collections import defaultdict

def init_memory():
    return {
        "facts": "",
        "style": ""
    }

memories = defaultdict(init_memory)

In [14]:
def run_recursive_summarization(dataset):
    memories = defaultdict(init_memory)
    augmented_dataset = []

    for sample in dataset:
        speaker = sample["target_speaker"]
        interaction_text = build_interaction_text(sample)

        # --- STUB OUTPUT (for now) ---
        raw_memory_update = {
            "facts": "The speaker sells products at cafes and mentions locations.",
            "style": "The speaker responds in a friendly and humorous manner."
        }

        # --- Stabilize + clamp ---
        for k in raw_memory_update:
            raw_memory_update[k] = clamp_text(raw_memory_update[k])
            raw_memory_update[k] = stabilize_memory(
                memories[speaker][k],
                raw_memory_update[k]
            )

        memories[speaker] = raw_memory_update

        # Attach structured memory
        new_sample = sample.copy()
        new_sample["memory_summary"] = memories[speaker]

        augmented_dataset.append(new_sample)

    return augmented_dataset, memories

In [15]:
test_data = data[:3]
augmented_data, final_memories = run_recursive_summarization(test_data)

for d in augmented_data:
    print("=" * 60)
    print("Speaker:", d["target_speaker"])
    print("Memory:", d["memory_summary"])


Speaker: nickeymousee
Memory: {'facts': 'The speaker sells products at cafes and mentions locations', 'style': 'The speaker responds in a friendly and humorous manner'}
Speaker: Soggypaper_
Memory: {'facts': 'The speaker sells products at cafes and mentions locations', 'style': 'The speaker responds in a friendly and humorous manner'}
Speaker: Soggypaper_
Memory: {'facts': 'The speaker sells products at cafes and mentions locations', 'style': 'The speaker responds in a friendly and humorous manner'}


# Speaker-wise Chronological Ordering

In [16]:
from collections import defaultdict

def group_by_speaker(dataset):
    grouped = defaultdict(list)
    for sample in dataset:
        grouped[sample["target_speaker"]].append(sample)
    return grouped


In [17]:
def sort_speaker_samples(grouped_samples):
    for speaker in grouped_samples:
        grouped_samples[speaker].sort(
            key=lambda x: x["s_node"]["turn_index"]
        )
    return grouped_samples


In [18]:
def flatten_grouped_samples(grouped_samples):
    ordered_dataset = []
    for speaker in grouped_samples:
        ordered_dataset.extend(grouped_samples[speaker])
    return ordered_dataset


In [19]:
def reorder_dataset_by_speaker_time(dataset):
    grouped = group_by_speaker(dataset)
    grouped = sort_speaker_samples(grouped)
    return flatten_grouped_samples(grouped)


In [20]:
ordered_data = reorder_dataset_by_speaker_time(data)

test_data = ordered_data[:5]
augmented_data, final_memories = run_recursive_summarization(test_data)

for d in augmented_data:
    print("=" * 60)
    print("Speaker:", d["target_speaker"])
    print("Turn index:", d["s_node"]["turn_index"])
    print("Memory:", d["memory_summary"])


Speaker: nickeymousee
Turn index: 2
Memory: {'facts': 'The speaker sells products at cafes and mentions locations', 'style': 'The speaker responds in a friendly and humorous manner'}
Speaker: nickeymousee
Turn index: 2
Memory: {'facts': 'The speaker sells products at cafes and mentions locations', 'style': 'The speaker responds in a friendly and humorous manner'}
Speaker: nickeymousee
Turn index: 2
Memory: {'facts': 'The speaker sells products at cafes and mentions locations', 'style': 'The speaker responds in a friendly and humorous manner'}
Speaker: nickeymousee
Turn index: 2
Memory: {'facts': 'The speaker sells products at cafes and mentions locations', 'style': 'The speaker responds in a friendly and humorous manner'}
Speaker: nickeymousee
Turn index: 4
Memory: {'facts': 'The speaker sells products at cafes and mentions locations', 'style': 'The speaker responds in a friendly and humorous manner'}


# Persist the Augmented Dataset

In [21]:
import json

def save_jsonl(dataset, path):
    """
    Saves a list of dicts as JSON Lines (JSONL).
    """
    with open(path, "w", encoding="utf-8") as f:
        for sample in dataset:
            f.write(json.dumps(sample, ensure_ascii=False) + "\n")


In [22]:
# 1. Reorder dataset by speaker/time
ordered_data = reorder_dataset_by_speaker_time(data)

# 2. Run recursive summarization
augmented_data, final_memories = run_recursive_summarization(ordered_data)

# 3. Save to disk
output_path = "mpchat_recursive_memory_stub.jsonl"
save_jsonl(augmented_data, output_path)

print(f"Saved {len(augmented_data)} samples to {output_path}")


Saved 7007 samples to mpchat_recursive_memory_stub.jsonl


# Loading in DeepSeek

In [23]:
import torch
print(torch.__version__)

2.5.1+cu121


In [24]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "mistralai/Mistral-7B-v0.1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): MistralRMSNorm((4096,)

## Altered version on llm call replacing the stub

In [25]:
def build_memory_prompt(speaker, previous_memory, interaction_text):
    return (
        f"Speaker: {speaker}\n\n"
        f"Previous memory: {previous_memory}\n\n"
        f"New interaction:\n"
        f"{interaction_text}\n\n"
        f"Updated memory (JSON):\n"
        f'{{"facts":"", "style":""}}'
    )


In [46]:
def call_llm(prompt, max_new_tokens=120):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,      # deterministic for memory
            temperature=0.0,
            top_p=1.0,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

        decoded = tokenizer.decode(
        output[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()
    
    # 🔴 CRITICAL FIX
    if not decoded.startswith("{"):
        decoded = "{" + decoded


    return decoded.strip()


In [27]:
import json
import re

def safe_parse_memory(text):
    """
    Extract the LAST JSON object that contains at least 'facts' and 'style'.
    Ignore everything else.
    """
    candidates = re.findall(r"\{[\s\S]*?\}", text)

    for block in reversed(candidates):
        try:
            obj = json.loads(block)
            if "facts" in obj and "style" in obj:
                return {
                    "facts": obj["facts"].strip(),
                    "style": obj["style"].strip()
                }
        except Exception:
            continue

    # Fallback if nothing valid found
    return {"facts": "", "style": ""}

In [47]:
def sanitize_facts(facts, speaker_reply):
    if facts.lower() in speaker_reply.lower():
        return facts
    return ""


In [49]:
print(call_llm("Describe the speaker's personality in one sentence."))

Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.


Describe the poem's theme in one sentence.
The speaker is a person who is very self-conscious and insecure about their appearance, and the theme of the poem is the struggle to find acceptance and love in a world that values physical beauty.


In [28]:
prompt = build_memory_prompt(
    speaker=data[0]["target_speaker"],
    previous_memory="",
    interaction_text=build_interaction_text(data[0])
)

print(prompt)

Speaker: nickeymousee

Previous memory: 

New interaction:
Other user (stylofylo) said
(sentiment toward nickeymousee: positive):
"omg i work at the cafe where we sell ur pins! they're so cute"

nickeymousee replied:
"which cafe? i only have them at trueshot right now and in edmonton soon edit: saw a comment for hamilton haha, so cool"

Updated memory (JSON):
{"facts":"", "style":""}


### Trying the llm with one prompt

In [34]:
prompt = build_memory_prompt(
    speaker=data[0]["target_speaker"],
    previous_memory='{"facts": "", "style": ""}',
    interaction_text=build_interaction_text(data[0])
)

print(prompt) 

raw_text = call_llm(prompt)
print("RAW OUTPUT:\n", raw_text)

parsed = safe_parse_memory(raw_text)
print("\nPARSED MEMORY:\n", parsed)


SYSTEM:
You are a function that outputs JSON only.

USER:
Maintain a long-term memory about the speaker "nickeymousee".

Current memory (JSON):
{"facts": "", "style": ""}

New interaction:
Other user (stylofylo) said
(sentiment toward nickeymousee: positive):
"omg i work at the cafe where we sell ur pins! they're so cute"

nickeymousee replied:
"which cafe? i only have them at trueshot right now and in edmonton soon edit: saw a comment for hamilton haha, so cool"

TASK:
Extract persona information from the new interaction and update the memory.
You MAY extract facts that are directly stated or clearly implied by the speaker's own reply.
Do NOT describe the update process itself.


STRICT OUTPUT REQUIREMENTS:
- Output ONLY valid JSON
- Output EXACTLY two keys: "facts" and "style"
- Facts explicitly mentioned by the speaker (e.g., selling items, locations, work) are NOT considered invented
- Do NOT include explanations
- Do NOT include examples
- Do NOT include markdown
- Do NOT include

# Trying out the pipeline for one user

In [35]:
TARGET_USER = "nickeymousee"


In [36]:
user_data = [
    sample for sample in data
    if sample["target_speaker"] == TARGET_USER
]

print(f"Found {len(user_data)} samples for {TARGET_USER}")

Found 6 samples for nickeymousee


In [37]:
user_data = sorted(
    user_data,
    key=lambda x: x["s_node"]["turn_index"]
)

In [38]:
user_data

[{'dialog_id': '6ee8654c-4189-49bf-b272-6c6f9a1cb99d',
  'target_speaker': 'nickeymousee',
  'context': [{'turn_index': 1,
    'speaker_id': 'stylofylo',
    'utterance': "omg i work at the cafe where we sell ur pins! they're so cute",
    'sentiment_towards_target': 'positive',
    'debug': {'pos_score': 1.0,
     'neg_score': 0.5,
     'alice_sentiment': 0.7177,
     'similarity': 0.35043150186538696}}],
  's_node': {'turn_index': 2,
   'speaker_id': 'nickeymousee',
   'utterance': 'which cafe? i only have them at trueshot right now and in edmonton soon edit: saw a comment for hamilton haha, so cool'}},
 {'dialog_id': '79902a9a-45de-4299-9398-9223c2d85d8a',
  'target_speaker': 'nickeymousee',
  'context': [{'turn_index': 1,
    'speaker_id': 'BitesBytes84',
    'utterance': 'are you going to design chemex pins as well?',
    'sentiment_towards_target': 'negative',
    'debug': {'pos_score': 1.0,
     'neg_score': 1.0,
     'alice_sentiment': 0.8197,
     'similarity': 0.2972974479198

In [44]:
memory = {"facts": "", "style": ""}

sample = user_data[0]  # nickeymousee first interaction
interaction_text = build_interaction_text(sample)

prompt = build_memory_prompt(
    speaker=TARGET_USER,
    previous_memory=json.dumps(memory),
    interaction_text=interaction_text
)

raw = call_llm(prompt)
parsed = safe_parse_memory(raw)

print("RAW:", raw)
print("PARSED:", parsed)


KeyboardInterrupt: 

In [50]:
def filter_non_speaker_facts(facts, speaker_reply):
    facts_lower = facts.lower()
    reply_lower = speaker_reply.lower()

    # require lexical overlap
    overlap = any(
        token in reply_lower
        for token in facts_lower.split()
        if len(token) > 4
    )

    return facts if overlap else ""


In [51]:
INVALID_STYLES = {"positive", "negative", "neutral"}

def normalize_style(style):
    if style.lower() in INVALID_STYLES:
        return ""
    return style


In [ ]:
memory = {"facts": "", "style": ""}

for i, sample in enumerate(user_data):
    print("\n" + "=" * 80)
    print(f"STEP {i+1}")
    print(f"Turn index: {sample['s_node']['turn_index']}")

    interaction_text = build_interaction_text(sample)

    prompt = build_memory_prompt(
        speaker=TARGET_USER,
        previous_memory=json.dumps(memory),
        interaction_text=interaction_text
    )

    raw_text = call_llm(prompt)
    parsed_memory = safe_parse_memory(raw_text)

    # --------------------------------------------------
    # 🔴 NEW: Extract speaker reply ONCE (ground truth)
    # --------------------------------------------------
    speaker_reply = sample["s_node"]["utterance"]

    # --------------------------------------------------
    # 🔴 NEW: Guard 1 — speaker-only facts
    # --------------------------------------------------
    facts = filter_non_speaker_facts(
        parsed_memory.get("facts", ""),
        speaker_reply
    )

    # --------------------------------------------------
    # 🔴 NEW: Guard 2 — sentiment ≠ style
    # --------------------------------------------------
    style = normalize_style(parsed_memory.get("style", ""))

    # --------------------------------------------------
    # 🔴 EXISTING: clamp + stabilize (NOW applied correctly)
    # --------------------------------------------------
    facts = clamp_text(facts)
    facts = stabilize_memory(memory["facts"], facts)

    style = clamp_text(style)
    style = stabilize_memory(memory["style"], style)

    # --------------------------------------------------
    # ✅ Update memory
    # --------------------------------------------------
    memory = {
        "facts": facts,
        "style": style
    }

    print("\nINTERACTION:")
    print(interaction_text)

    print("\nRAW MODEL OUTPUT:")
    print(raw_text)

    print("\nUPDATED MEMORY:")
    print(memory)



STEP 1
Turn index: 2


In [115]:
memory

{'facts': '', 'style': ''}

In [117]:
prompt = (
    'Speaker: nickeymousee\n\n'
    'Previous memory: {"facts": "", "style": ""}\n\n'
    'New interaction:\n'
    'Other user (stylofylo): "omg i work at the cafe where we sell ur pins! they\'re so cute"\n'
    'nickeymousee: "which cafe? i only have them at trueshot right now and in edmonton soon edit: saw a comment for hamilton haha, so cool"\n\n'
    'Updated memory (JSON):\n'
    '{"facts":"", "style":""}\n'
)


In [118]:
print("Model:", MODEL_NAME)
print("Tokenizer revision:", tokenizer._tokenizer.model.__class__.__name__)
print("Model config:", model.config._name_or_path)

Model: deepseek-ai/deepseek-llm-7b-chat
Tokenizer revision: BPE
Model config: deepseek-ai/deepseek-llm-7b-chat


In [119]:
print(call_llm(prompt))

{"facts": "nickeymousee has pins available at trueshot and edmonton, and possibly hamilton.", "style": "The interaction between nickeymousee and stylofylo took place at a cafe where nickeymousee's pins are sold.", "style_notes": "Stylofylo works at the cafe where nickeymousee's pins are sold."}
